# IRENA CAPEX Data Cleaning

تنظيف بيانات تكلفة إنشاء محطات الطاقة الشمسية والرياح من ملف IRENA.

**البيانات المستخدمة فقط:**
- Solar PV: Sheet `Fig 3.1`
- Onshore Wind: Sheet `Fig 2.3`
- Weighted average فقط
- السنوات من 2015 إلى 2025
- الوحدة: `2025 USD/kW`

شغّل الخلايا بالترتيب.

## Cell 1 — Import libraries

In [ ]:
import pandas as pd

print("Libraries imported successfully")

## Cell 2 — Read the Excel file

In [ ]:
file_path = r"E:\DEPI project DATA\prices\IRENA_TEC_RPGC_in_2025_data_file_2026.xlsx"

# Read the two required sheets without using the first row as headers
solar_raw = pd.read_excel(file_path, sheet_name="Fig 3.1", header=None)
wind_raw = pd.read_excel(file_path, sheet_name="Fig 2.3", header=None)

print("Solar sheet shape:", solar_raw.shape)
print("Wind sheet shape:", wind_raw.shape)

## Cell 3 — Extract Solar weighted average

نأخذ صف `Weighted average` فقط، والسنوات من 2015 فما بعد.

In [ ]:
# Years are in row 5, starting from column 3
years = solar_raw.iloc[4, 2:].astype(int)

# Weighted average is in row 7
solar_values = solar_raw.iloc[6, 2:]

solar = pd.DataFrame({
    "Year": years,
    "Technology": "Solar",
    "Cost_USD_per_kW": pd.to_numeric(solar_values, errors="coerce")
})

# Keep 2015 onwards
solar = solar[solar["Year"] >= 2015].copy()

solar.head()

## Cell 4 — Extract Onshore Wind weighted average

نفس الخطوات، لكن من Sheet `Fig 2.3`.

In [ ]:
# Years are in row 5, starting from column 3
years = wind_raw.iloc[4, 2:].astype(int)

# Weighted average is in row 7
wind_values = wind_raw.iloc[6, 2:]

wind = pd.DataFrame({
    "Year": years,
    "Technology": "Onshore Wind",
    "Cost_USD_per_kW": pd.to_numeric(wind_values, errors="coerce")
})

# Keep 2015 onwards
wind = wind[wind["Year"] >= 2015].copy()

wind.head()

## Cell 5 — Combine Solar and Wind

In [ ]:
# Combine both technologies into one table
cleaned_data = pd.concat([solar, wind], ignore_index=True)

# Sort by year and technology
cleaned_data = cleaned_data.sort_values(
    by=["Year", "Technology"]
).reset_index(drop=True)

# Check missing values and duplicates
print("Missing values:")
print(cleaned_data.isna().sum())

print("\nDuplicate rows:", cleaned_data.duplicated().sum())

print("\nCleaned data:")
display(cleaned_data)

## Cell 6 — Save the cleaned file

In [ ]:
output_path = r"E:\DEPI project DATA\prices\Renewable_CAPEX_Cleaned.csv"

cleaned_data.to_csv(output_path, index=False)

print("File saved successfully:")
print(output_path)

## Cell 7 — Quick check

نتأكد إن البيانات فيها السنوات المطلوبة فقط، والتقنيتين موجودتين.

In [ ]:
print("Years:", cleaned_data["Year"].min(), "to", cleaned_data["Year"].max())
print("Technologies:", cleaned_data["Technology"].unique())
print("Number of rows:", len(cleaned_data))
print("Unit: 2025 USD per kW")